# Task 02: Eye Gaze Detection

Detect where the student's EYES are looking — even if head stays forward.

A student can keep head straight but peek with eyes. This catches that.

**Press 'q' to quit.**

## Cell 1: Setup

Load MediaPipe Face Landmarker — same model as head direction, but now we use the IRIS landmarks.

In [1]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
import numpy as np
import time

# Load Face Landmarker
model_path = '../01_head_direction/face_landmarker.task'

base_options = mp_python.BaseOptions(model_asset_path=model_path)
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    num_faces=1,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False,
    running_mode=vision.RunningMode.IMAGE
)
landmarker = vision.FaceLandmarker.create_from_options(options)

print("Face Landmarker loaded!")
print("Using IRIS landmarks for eye gaze tracking.")

Face Landmarker loaded!
Using IRIS landmarks for eye gaze tracking.


## Cell 2: Configuration

In [2]:
# =============================================
# CONFIGURATION
# =============================================

# How far iris must shift to count as "looking sideways"
# Lower = more sensitive, Higher = less sensitive
GAZE_THRESHOLD = 0.25       # Try 0.15 (strict) to 0.35 (loose)

# How long eyes must be off-center to be suspicious
GAZE_SUSPICIOUS_DURATION = 2.0   # seconds

# Head direction thresholds (same as Task 01)
HEAD_TURN_THRESHOLD = 0.15
HEAD_UP_THRESHOLD = 0.1
HEAD_SUSPICIOUS_DURATION = 3.0

print("Configuration:")
print(f"  Gaze threshold:   {GAZE_THRESHOLD} (iris shift ratio)")
print(f"  Gaze suspicious:  {GAZE_SUSPICIOUS_DURATION}s")
print(f"  Head threshold:   {HEAD_TURN_THRESHOLD}")
print(f"  Head suspicious:  {HEAD_SUSPICIOUS_DURATION}s")

Configuration:
  Gaze threshold:   0.25 (iris shift ratio)
  Gaze suspicious:  2.0s
  Head threshold:   0.15
  Head suspicious:  3.0s


## Cell 3: Detection Functions

Two functions:
1. `get_head_direction()` — same as Task 01
2. `get_eye_gaze()` — NEW: tracks iris position inside the eye

### How eye gaze works:
```
MediaPipe gives us these landmarks per eye:
  - Left corner of eye
  - Right corner of eye  
  - Iris center (pupil position)

We calculate: iris position relative to eye corners
  Iris at 50% between corners = looking CENTER
  Iris at 20% (closer to left corner) = looking LEFT
  Iris at 80% (closer to right corner) = looking RIGHT
```

In [3]:
def get_head_direction(face_landmarks, width, height):
    """Same as Task 01 — detect head turn direction."""
    nose = face_landmarks[4]
    left_eye = face_landmarks[133]
    right_eye = face_landmarks[362]
    
    nose_x = nose.x * width
    left_eye_x = left_eye.x * width
    right_eye_x = right_eye.x * width
    eye_center_x = (left_eye_x + right_eye_x) / 2
    eye_center_y = (left_eye.y + right_eye.y) / 2 * height
    nose_y = nose.y * height
    
    eye_distance = abs(right_eye_x - left_eye_x)
    if eye_distance == 0:
        return "UNKNOWN", 0, 0
    
    h_ratio = (nose_x - eye_center_x) / eye_distance
    v_ratio = (nose_y - eye_center_y) / eye_distance
    
    if v_ratio < -HEAD_UP_THRESHOLD:
        return "UP", h_ratio, v_ratio
    elif h_ratio < -HEAD_TURN_THRESHOLD:
        return "LEFT", h_ratio, v_ratio
    elif h_ratio > HEAD_TURN_THRESHOLD:
        return "RIGHT", h_ratio, v_ratio
    else:
        return "FORWARD", h_ratio, v_ratio


def get_eye_gaze(face_landmarks, width, height):
    """
    Detect where the eyes are looking by tracking iris position.
    
    Key landmarks:
      Left eye:  inner corner=133, outer corner=33, iris center=468
      Right eye: inner corner=362, outer corner=263, iris center=473
    
    Returns: (gaze_direction, left_ratio, right_ratio)
    """
    
    # LEFT EYE iris tracking
    left_eye_inner = face_landmarks[133]    # Inner corner (near nose)
    left_eye_outer = face_landmarks[33]     # Outer corner (near ear)
    left_iris = face_landmarks[468]          # Iris center
    
    # RIGHT EYE iris tracking
    right_eye_inner = face_landmarks[362]   # Inner corner (near nose)
    right_eye_outer = face_landmarks[263]   # Outer corner (near ear)
    right_iris = face_landmarks[473]         # Iris center
    
    # Calculate iris position as ratio within eye (0=outer corner, 1=inner corner)
    # Left eye
    left_eye_width = abs(left_eye_inner.x - left_eye_outer.x)
    if left_eye_width == 0:
        left_ratio = 0.5
    else:
        left_ratio = (left_iris.x - left_eye_outer.x) / left_eye_width
    
    # Right eye
    right_eye_width = abs(right_eye_inner.x - right_eye_outer.x)
    if right_eye_width == 0:
        right_ratio = 0.5
    else:
        right_ratio = (right_iris.x - right_eye_outer.x) / right_eye_width
    
    # Average both eyes for more stable reading
    avg_ratio = (left_ratio + right_ratio) / 2
    
    # Center is around 0.5, deviation from center = looking sideways
    # After flip: lower ratio = looking right, higher ratio = looking left
    deviation = avg_ratio - 0.5  # negative = one side, positive = other side
    
    if deviation < -GAZE_THRESHOLD:
        gaze = "RIGHT"
    elif deviation > GAZE_THRESHOLD:
        gaze = "LEFT"
    else:
        gaze = "CENTER"
    
    return gaze, left_ratio, right_ratio, avg_ratio


print("Both detection functions ready!")
print("  get_head_direction() — tracks head turn")
print("  get_eye_gaze() — tracks iris/pupil position")

Both detection functions ready!
  get_head_direction() — tracks head turn
  get_eye_gaze() — tracks iris/pupil position


## Cell 4: Live Detection — Head Direction + Eye Gaze Combined

**What you'll see on screen:**
- Top bar: Head direction (FORWARD/LEFT/RIGHT/UP)
- Second bar: Eye gaze (CENTER/LEFT/RIGHT)
- Iris dots drawn on your eyes (cyan dots)
- Combined verdict at bottom

**Test these scenarios:**
- Look straight → Head: FORWARD, Eyes: CENTER (all green)
- Turn head left → Head: LEFT, Eyes: follow (both flag)
- Keep head straight but look left with ONLY your eyes → Head: FORWARD, Eyes: LEFT (caught!)

**Press 'q' to quit.**

In [ ]:
camera = cv2.VideoCapture(0)
camera.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
camera.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

if not camera.isOpened():
    print("ERROR: Cannot open webcam!")
else:
    print("Head + Eye Gaze Detection running!")
    print("NO calibration needed — system learns automatically.")
    print("Press 'q' to quit.\n")
    
    # ROLLING CALIBRATION — no user action needed!
    # Keeps last 100 readings. The MEDIAN = your natural "center"
    # Why median not average? Because if you peek left for 5 seconds,
    # average shifts. Median ignores outliers = more stable.
    gaze_history = []
    HISTORY_SIZE = 100          # Last 100 readings (~3 seconds of data)
    YOUR_CENTER = 0.5           # Starting guess, updates automatically
    MIN_READINGS = 15           # Need at least 15 readings before detecting
    
    head_turn_start = None
    gaze_turn_start = None
    frame_count = 0
    start_time = time.time()
    
    while True:
        success, frame = camera.read()
        if not success:
            break
        
        frame = cv2.flip(frame, 1)
        frame_count += 1
        height, width = frame.shape[:2]
        
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        results = landmarker.detect(mp_image)
        
        if results.face_landmarks and len(results.face_landmarks) > 0:
            face_lms = results.face_landmarks[0]
            
            # HEAD DIRECTION
            head_dir, h_ratio, v_ratio = get_head_direction(face_lms, width, height)
            
            # EYE GAZE (raw ratio)
            _, l_ratio, r_ratio, avg_ratio = get_eye_gaze(face_lms, width, height)
            
            # ADD to rolling history
            gaze_history.append(avg_ratio)
            if len(gaze_history) > HISTORY_SIZE:
                gaze_history.pop(0)  # Remove oldest reading
            
            # UPDATE center using MEDIAN of history
            # Median = the middle value. Ignores extreme peeks.
            if len(gaze_history) >= MIN_READINGS:
                YOUR_CENTER = np.median(gaze_history)
            
            # Calculate deviation from YOUR rolling center
            deviation = avg_ratio - YOUR_CENTER
            
            if len(gaze_history) < MIN_READINGS:
                gaze_dir = "LEARNING"  # Not enough data yet
            elif deviation < -GAZE_THRESHOLD:
                gaze_dir = "RIGHT"
            elif deviation > GAZE_THRESHOLD:
                gaze_dir = "LEFT"
            else:
                gaze_dir = "CENTER"
            
            # DRAW IRIS + EYE CORNERS
            for iris_idx in [468, 473]:
                ix = int(face_lms[iris_idx].x * width)
                iy = int(face_lms[iris_idx].y * height)
                cv2.circle(frame, (ix, iy), 4, (255, 255, 0), -1)
            for corner_idx in [33, 133, 263, 362]:
                cx = int(face_lms[corner_idx].x * width)
                cy = int(face_lms[corner_idx].y * height)
                cv2.circle(frame, (cx, cy), 2, (0, 255, 0), -1)
            
            # HEAD STATUS BAR
            if head_dir == "FORWARD":
                head_color = (0, 180, 0)
                head_status = "Head: FORWARD"
                head_turn_start = None
            else:
                if head_turn_start is None:
                    head_turn_start = time.time()
                dur = time.time() - head_turn_start
                if dur < HEAD_SUSPICIOUS_DURATION:
                    head_color = (0, 180, 200)
                    head_status = f"Head: {head_dir} ({dur:.1f}s)"
                else:
                    head_color = (0, 0, 200)
                    head_status = f"Head: {head_dir} ({dur:.1f}s) !!"
            
            cv2.rectangle(frame, (0, 0), (width, 35), head_color, -1)
            cv2.putText(frame, head_status, (10, 25),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
            
            # EYE GAZE STATUS BAR
            if gaze_dir == "LEARNING":
                gaze_color = (200, 100, 0)
                gaze_status = f"Eyes: Learning... ({len(gaze_history)}/{MIN_READINGS})"
                gaze_turn_start = None
            elif gaze_dir == "CENTER":
                gaze_color = (0, 180, 0)
                gaze_status = "Eyes: CENTER"
                gaze_turn_start = None
            else:
                if gaze_turn_start is None:
                    gaze_turn_start = time.time()
                dur = time.time() - gaze_turn_start
                if dur < GAZE_SUSPICIOUS_DURATION:
                    gaze_color = (0, 180, 200)
                    gaze_status = f"Eyes: {gaze_dir} ({dur:.1f}s)"
                else:
                    gaze_color = (0, 0, 200)
                    gaze_status = f"Eyes: {gaze_dir} ({dur:.1f}s) !!"
            
            cv2.rectangle(frame, (0, 35), (width, 70), gaze_color, -1)
            cv2.putText(frame, gaze_status, (10, 60),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
            
            # COMBINED VERDICT
            head_ok = head_dir == "FORWARD"
            eyes_ok = gaze_dir == "CENTER" or gaze_dir == "LEARNING"
            
            if head_ok and eyes_ok:
                verdict = "ALL CLEAR - Student focused"
                v_color = (0, 200, 0)
            elif not head_ok and not eyes_ok:
                verdict = "HIGH SUSPICION - Head AND eyes away!"
                v_color = (0, 0, 255)
            elif not eyes_ok:
                verdict = "SUSPICIOUS - Eyes peeking!"
                v_color = (0, 100, 255)
            else:
                verdict = "WARNING - Head turned"
                v_color = (0, 180, 255)
            
            cv2.rectangle(frame, (0, height-40), (width, height), v_color, -1)
            cv2.putText(frame, verdict, (10, height-15),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
            
            # Debug
            cv2.putText(frame, f"Iris:{avg_ratio:.3f} Center:{YOUR_CENTER:.3f} Dev:{deviation:+.3f}",
                       (10, height-50), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (200,200,200), 1)
        
        else:
            cv2.rectangle(frame, (0, 0), (width, 70), (100,100,100), -1)
            cv2.putText(frame, "No face detected", (10, 45),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)
        
        elapsed = time.time() - start_time
        fps = frame_count / elapsed if elapsed > 0 else 0
        cv2.putText(frame, f"FPS:{fps:.0f}", (width-80, height-50),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200,200,200), 1)
        
        cv2.imshow('ExamGuard - Head + Eyes (q to quit)', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    camera.release()
    cv2.destroyAllWindows()
    print(f"Done! {elapsed:.0f}s, {fps:.0f} FPS")

Head + Eye Gaze Detection running!
NO calibration needed — system learns automatically.
Press 'q' to quit.



## What to Test

| You do | Head says | Eyes say | Verdict |
|--------|----------|---------|--------|
| Look straight at screen | FORWARD | CENTER | ALL CLEAR (green) |
| Turn head left | LEFT | follows | HIGH SUSPICION (red) |
| Head straight, peek left with eyes only | FORWARD | LEFT | SUSPICIOUS - eyes peeking (orange) |
| Head straight, peek right with eyes only | FORWARD | RIGHT | SUSPICIOUS - eyes peeking (orange) |
| Look up | UP | CENTER | WARNING - head turned (yellow) |

**The key test:** Keep your head PERFECTLY still, look straight, then slowly move ONLY your eyes to the side. The system should catch it!

### Tuning:
- Eyes too sensitive? → Increase `GAZE_THRESHOLD` (e.g., 0.3)
- Missing eye peeks? → Decrease `GAZE_THRESHOLD` (e.g., 0.15)
- Go back to Cell 2, change, re-run Cell 4

## Next Step

Task 03: Body Turning — detect shoulder rotation toward neighbor.